# Notebook 4: Analyzing a LangGraph Pipeline

Import a LangGraph `StateGraph` directly into `minimal-oversight` and
get a full governance analysis — no manual graph construction needed.

**What this notebook shows:**
1. Define a LangGraph workflow (or mock one without installing langgraph)
2. Import it with `from_langgraph()` — auto role inference, edge extraction
3. Run `analyze_pipeline()` — feasibility, masking, bottlenecks
4. Feed LangSmith traces for calibration
5. Compare: before vs after trace-based calibration

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

from minimal_oversight import analyze_pipeline
from minimal_oversight.connectors.langgraph import from_langgraph, normalize_langgraph
from minimal_oversight.connectors.traces import from_langsmith_traces, to_workflow_traces
from minimal_oversight.schema import NodeRole

import numpy as np

## 1. Build a real LangGraph workflow

**Scenario:** A research assistant pipeline:
- `researcher` — searches and gathers information
- `writer` — drafts a response from research results
- `fact_checker` — validates claims against sources
- `editor` — polishes the final output

In [ ]:
class ResearchState(TypedDict):
    query: str
    content: str

def research_documents(state):
    """Searches knowledge base and retrieves relevant documents."""
    return state

def write_response(state):
    """Drafts a comprehensive response from research findings."""
    return state

def check_facts(state):
    """Validates factual claims against source documents."""
    return state

def edit_and_polish(state):
    """Reviews grammar, tone, and completeness."""
    return state

# Build the LangGraph workflow
graph = StateGraph(ResearchState)
graph.add_node("researcher", research_documents)
graph.add_node("writer", write_response)
graph.add_node("fact_checker", check_facts)
graph.add_node("editor", edit_and_polish)
graph.add_edge("researcher", "writer")
graph.add_edge("writer", "fact_checker")
graph.add_edge("fact_checker", "editor")
graph.add_edge("editor", END)
graph.set_entry_point("researcher")

compiled = graph.compile()
print(f"Compiled LangGraph: {type(compiled).__name__}")
print(f"Nodes: {list(compiled.builder.nodes.keys())}")

## 2. Import into minimal-oversight

One call — the connector extracts nodes, edges, infers roles from
function names and docstrings, and assigns default parameters.

In [ ]:
# Inspect the normalized representation
normalized = normalize_langgraph(compiled)

print("Normalized pipeline:")
print(f"  Framework: {normalized.framework_source}")
print(f"  Nodes: {len(normalized.nodes)}")
print(f"  Edges: {len(normalized.edges)}")
print()

for n in normalized.nodes:
    print(f"  {n.id:<20} role={n.role.value:<12} desc=\"{n.description}\"")
print()
for e in normalized.edges:
    print(f"  {e.source_id} → {e.target_id}")

In [ ]:
# Convert to PipelineGraph
pipeline = from_langgraph(compiled)
print(pipeline)
print(f"Depth: {pipeline.depth}")
print(f"Sources: {pipeline.sources()}")
print(f"Sinks: {pipeline.sinks()}")
print()

# Check inferred parameters
print(f"{'Node':<20} {'Role':>12} {'σ_skill':>10} {'catch_rate':>12} {'K/N':>8}")
print("-" * 65)
for name, node in pipeline.nodes.items():
    role = node.metadata.get('role', 'unknown')
    print(f"{name:<20} {role:>12} {node.sigma_skill:>10.2f} {node.catch_rate:>12.2f} {node.review_capacity:>8.2f}")

## 3. Full analysis — one call

In [ ]:
report = analyze_pipeline(pipeline, p_min=0.70)
print(report)

## 4. Auto-detect: pass the compiled graph directly

`analyze_pipeline()` auto-detects real LangGraph objects by module name.

In [ ]:
# Pass the compiled LangGraph directly — auto-detected
report_auto = analyze_pipeline(compiled, p_min=0.70)
print(f"Auto-detected: C_op={report_auto.feasibility.c_op:.3f}")
print(f"Same result as explicit import: {np.isclose(report_auto.feasibility.c_op, report.feasibility.c_op)}")

## 5. Override parameters for nodes you know better

Role inference bootstraps defaults, but you know your agents better.
Override specific nodes with measured or estimated values.

In [ ]:
# You know the researcher uses RAG and is strong, but the writer struggles
calibrated = from_langgraph(mock_graph, parameter_overrides={
    "researcher": {"sigma_skill": 0.80, "catch_rate": 0.50},
    "writer":     {"sigma_skill": 0.45, "catch_rate": 0.70},
    "fact_checker":{"sigma_skill": 0.70, "catch_rate": 0.80},
    "editor":     {"sigma_skill": 0.65, "catch_rate": 0.75},
})

calibrated_report = analyze_pipeline(calibrated, p_min=0.70)

print("With calibrated parameters:")
print(f"  C_op = {calibrated_report.feasibility.c_op:.3f} (was {report.feasibility.c_op:.3f} with defaults)")
print(f"  Feasible: {calibrated_report.feasibility.feasible}")
print(f"  Bottleneck: {calibrated_report.feasibility.bottleneck_node}")
print()
for r in calibrated_report.recommendations[:3]:
    print(f"  {r.priority}. [{r.target_node}] {r.action}")

## 6. Calibrate from LangSmith traces

In production, export your LangSmith runs and feed them to the trace parser.
Here we simulate what LangSmith trace data looks like.

In [ ]:
# Simulated LangSmith run exports
# In production: runs = langsmith_client.list_runs(project_name="my-project")
simulated_runs = []
rng = np.random.default_rng(42)

node_skills = {"researcher": 0.80, "writer": 0.55, "fact_checker": 0.70, "editor": 0.65}
node_errors = {"researcher": "search timeout", "writer": "quality below threshold",
               "fact_checker": "claim unverified", "editor": "style issues"}

for i in range(200):
    child_runs = []
    t = 0.0
    for node_name, skill in node_skills.items():
        success = rng.random() < skill
        child_runs.append({
            "name": node_name,
            "run_type": "chain" if node_name in ("researcher", "fact_checker") else "llm",
            "status": "success" if success else "error",
            "error": None if success else node_errors[node_name],
            "start_time": t,
            "end_time": t + 1.0,
            "child_runs": [],
        })
        t += 1.0

    run = {
        "id": f"run_{i:04d}",
        "name": "research_assistant",
        "run_type": "chain",
        "status": "success",
        "start_time": f"2026-03-{15 + i // 50:02d}T{10 + (i % 10)}:00:00Z",
        "end_time": f"2026-03-{15 + i // 50:02d}T{10 + (i % 10)}:05:00Z",
        "child_runs": child_runs,
    }
    simulated_runs.append(run)

print(f"Simulated {len(simulated_runs)} LangSmith runs.")

In [ ]:
# Parse traces
normalized_traces = from_langsmith_traces(simulated_runs)
workflow_traces = to_workflow_traces(normalized_traces)

print(f"Parsed {len(normalized_traces)} traces → {len(workflow_traces)} workflow traces")
print(f"First trace has {len(workflow_traces[0].node_outcomes)} node outcomes")

In [ ]:
# Re-analyze with trace data
trace_pipeline = from_langgraph(mock_graph)
trace_report = analyze_pipeline(trace_pipeline, p_min=0.70, traces=workflow_traces)

print("=== Defaults vs Trace-Calibrated ===")
print(f"\n{'Node':<20} {'Default σ_raw':>14} {'Traced σ_raw':>14} {'Traced M*':>10}")
print("-" * 62)

default_pipeline = from_langgraph(mock_graph)
default_report = analyze_pipeline(default_pipeline, p_min=0.70)

for name in trace_pipeline.topological_order():
    d_node = default_pipeline.get_node(name)
    t_est = trace_report.node_estimates.get(name)
    d_raw = d_node.sigma_raw or 0
    t_raw = t_est.sigma_raw if t_est else 0
    t_m = t_est.masking_index if t_est else 0
    print(f"{name:<20} {d_raw:>14.3f} {t_raw:>14.3f} {t_m:>10.2f}")

print(f"\nDefault C_op:  {default_report.feasibility.c_op:.3f}")
print(f"Traced C_op:   {trace_report.feasibility.c_op:.3f}")

## 7. Conditional routing

LangGraph workflows often have conditional edges — the connector extracts them too.

In [ ]:
def classify_query(state):
    """Classifies incoming query by type."""
    return state

def handle_simple(state):
    """Handles simple factual queries directly."""
    return state

def handle_complex(state):
    """Handles complex research queries with multiple steps."""
    return state

def quality_gate(state):
    """Final quality check before response delivery."""
    return state

# Build with real LangGraph conditional edges
routing_graph = StateGraph(ResearchState)
routing_graph.add_node("classifier", classify_query)
routing_graph.add_node("simple_handler", handle_simple)
routing_graph.add_node("complex_handler", handle_complex)
routing_graph.add_node("quality_gate", quality_gate)
routing_graph.set_entry_point("classifier")
routing_graph.add_conditional_edges(
    "classifier",
    lambda s: "simple",  # placeholder routing function
    {"simple": "simple_handler", "complex": "complex_handler"},
)
routing_graph.add_edge("simple_handler", "quality_gate")
routing_graph.add_edge("complex_handler", "quality_gate")
routing_graph.add_edge("quality_gate", END)

routing_compiled = routing_graph.compile()
routing_pipeline = from_langgraph(routing_compiled)

print(routing_pipeline)
print(f"Classifier children: {routing_pipeline.children('classifier')}")
print(f"Quality gate parents: {routing_pipeline.parents('quality_gate')}")
print()

routing_report = analyze_pipeline(routing_compiled, p_min=0.70)
print(f"C_op = {routing_report.feasibility.c_op:.3f}")
print(f"Feasible: {routing_report.feasibility.feasible}")
print()

# Motifs detected
for m in routing_report.motifs:
    print(f"[{m.motif.value}] {m.risk_description}")

## Key takeaways

1. **One import, full analysis** — `from_langgraph(compiled_graph)` extracts topology, infers roles, assigns defaults.
2. **Role inference bootstraps** — function names and docstrings classify nodes as generator/reviewer/router/etc. Override with `parameter_overrides` when you know better.
3. **Trace calibration is better** — LangSmith traces replace heuristic defaults with measured sigma_raw, sigma_corr, and catch rates.
4. **Conditional edges are extracted** — routing workflows with `add_conditional_edges` produce fan-out/merge motifs that the framework analyzes.
5. **Auto-detect works** — `analyze_pipeline(my_compiled_graph)` just works, no explicit connector call needed.